In [2]:
import pandas as pd
import numpy as np

# Generate a mock 100,000 row dataset mimicking the NYC Taxi schema
np.random.seed(42)
n_rows = 100000

mock_data = {
    'vendor_id': np.random.choice([1, 2, 3], n_rows),
    'passenger_count': np.random.randint(1, 6, n_rows),
    'trip_distance': np.random.exponential(scale=3.0, size=n_rows),
    'fare_amount': np.random.normal(loc=15.0, scale=10.0, size=n_rows).clip(2.5, 150.0),
    'tip_amount': np.random.exponential(scale=2.0, size=n_rows),
    'total_amount': np.zeros(n_rows)
}
# Make total_amount dependent
mock_data['total_amount'] = mock_data['fare_amount'] + mock_data['tip_amount']

# Create a secondary lookup table for the Join Benchmark
lookup_data = {
    'vendor_id': [1, 2, 3],
    'vendor_name': ['Creative', 'VeriFone', 'VTS_Inc']
}

pd.DataFrame(mock_data).to_parquet('sample_taxi.parquet')
pd.DataFrame(lookup_data).to_parquet('vendor_lookup.parquet')
print("Sample datasets successfully created locally!")

Sample datasets successfully created locally!


In [3]:
import os
import sys

# 1. Fix the PyArrow warning from the logs
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

# 2. Point PySpark to the correct Python executable inside your cloud environment
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["PYSPARK_SUBMIT_ARGS"] = "--master local[*] pyspark-shell"

# Now run your imports safely
import time
import pandas as pd
import dask.dataframe as dd
import pyspark.pandas as ps 

print("Frameworks imported successfully without errors!")

Frameworks imported successfully without errors!


In [4]:
import time
import pandas as pd
import dask.dataframe as dd
import pyspark.pandas as ps  # Modern Koalas replacement

# Initialize PySpark's Pandas Engine safely inside Vertex AI
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = "--master local[*] pyspark-shell"

def benchmark_dask(data_path, lookup_path):
    metrics = {}
    
    # 1. Read Data
    t0 = time.time()
    df = dd.read_parquet(data_path)
    df_lookup = dd.read_parquet(lookup_path)
    # Dask reads lazily; force compute on length to evaluate reading execution
    _ = len(df)
    metrics['1. Read Data'] = time.time() - t0
    
    # 2. Count Index (Simple count)
    t0 = time.time()
    _ = df.count().compute()
    metrics['2. Count Operation'] = time.time() - t0
    
    # 3. Complex Arithmetic Formula
    t0 = time.time()
    arithmetic_res = (df['fare_amount'] + df['tip_amount']) * 1.5 / (df['passenger_count'] + 1)
    _ = arithmetic_res.compute()
    metrics['3. Complex Arithmetic'] = time.time() - t0
    
    # 4. Statistical Standard Deviation
    t0 = time.time()
    _ = df['trip_distance'].std().compute()
    metrics['4. Standard Deviation'] = time.time() - t0
    
    # 5. GroupBy Aggregation
    t0 = time.time()
    _ = df.groupby('passenger_count')['fare_amount'].mean().compute()
    metrics['5. GroupBy Mean'] = time.time() - t0
    
    # 6. Merge/Join with Count
    t0 = time.time()
    joined = dd.merge(df, df_lookup, on='vendor_id', how='inner')
    _ = len(joined)
    metrics['6. Join & Count'] = time.time() - t0
    
    return metrics


def benchmark_koalas(data_path, lookup_path):
    metrics = {}
    
    # 1. Read Data
    t0 = time.time()
    df = ps.read_parquet(data_path)
    df_lookup = ps.read_parquet(lookup_path)
    _ = len(df)
    metrics['1. Read Data'] = time.time() - t0
    
    # 2. Count Index
    t0 = time.time()
    _ = df.count()  # Koalas acts closer to pandas but executes via Spark SQL Catalyst
    metrics['2. Count Operation'] = time.time() - t0
    
    # 3. Complex Arithmetic Formula
    t0 = time.time()
    arithmetic_res = (df['fare_amount'] + df['tip_amount']) * 1.5 / (df['passenger_count'] + 1)
    # Coerce action to trigger Spark evaluation
    _ = arithmetic_res.to_numpy() 
    metrics['3. Complex Arithmetic'] = time.time() - t0
    
    # 4. Statistical Standard Deviation
    t0 = time.time()
    _ = df['trip_distance'].std()
    metrics['4. Standard Deviation'] = time.time() - t0
    
    # 5. GroupBy Aggregation
    t0 = time.time()
    _ = df.groupby('passenger_count')['fare_amount'].mean()
    metrics['5. GroupBy Mean'] = time.time() - t0
    
    # 6. Merge/Join with Count
    t0 = time.time()
    joined = ps.merge(df, df_lookup, on='vendor_id', how='inner')
    _ = len(joined)
    metrics['6. Join & Count'] = time.time() - t0
    
    return metrics

# --- Execution Block ---
print("Running Dask Benchmark...")
dask_results = benchmark_dask('sample_taxi.parquet', 'vendor_lookup.parquet')

print("Running Koalas (PySpark) Benchmark...")
koalas_results = benchmark_koalas('sample_taxi.parquet', 'vendor_lookup.parquet')

# --- Formatting the Summary DataFrame ---
df_summary = pd.DataFrame({
    'Dask (seconds)': dask_results,
    'Koalas / PySpark (seconds)': koalas_results
})
df_summary['Ratio (Dask / Koalas)'] = df_summary['Dask (seconds)'] / df_summary['Koalas / PySpark (seconds)']

print("\n=== FINAL BENCHMARK SUMMARY TABLE ===")
display(df_summary)

Running Dask Benchmark...
Running Koalas (PySpark) Benchmark...


/home/jupyter/.micromamba/envs/bigdata_env/lib/python3.10/site-packages/pyspark/pandas/utils.py:1038: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_parquet`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/27 01:08:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
/home/jupyter/.micromamba/envs/bigdata_env/lib/python3.10/site-packages/pyspark/pandas/utils.py:1038: PandasAPIOnSparkAdviceWarning: `to_numpy` loads all data into the driver's memory. It should only be used if the resulting NumPy ndarray is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)



=== FINAL BENCHMARK SUMMARY TABLE ===


,Dask (seconds),Koalas / PySpark (seconds),Ratio (Dask / Koalas)
1. Read Data,0.172947,10.179116,0.016990
2. Count Operation,0.019258,0.983461,0.019582
3. Complex Arithmetic,0.017610,1.603103,0.010985
4. Standard Deviation,0.010334,0.414556,0.024927
5. GroupBy Mean,0.026094,0.150956,0.172862
6. Join & Count,0.026597,0.706702,0.037636
